# Questão 7 - Recomendation System

In [174]:
#Importando bibliotecas necessárias
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
#1. Crie uma matriz de interação Usuário × Produto obedecendo às regras abaixo:
#     a. Linhas: id_cliente
#     b. Colunas: id_produto
#     c. Valor da célula:
#     d. 1 se o cliente comprou ao menos uma vez o produto
#     e. 0 caso contrário
#     f. Ignore a quantidade comprada (presença/ ausência apenas)

#Importando datasets
products = pd.read_csv("Dataset/products.csv")
product_variants = pd.read_csv("Dataset/product_variants.csv")
orders = pd.read_csv("Dataset/orders.csv")
order_items = pd.read_csv("Dataset/order_items.csv")
customers = pd.read_csv("Dataset/customers.csv")

#Combinado os datasets por PK -> FK
df_combined_customers = (
    orders[["id", "customer_id"]]
    .merge(
        order_items[["order_id", "product_variant_id"]],
        left_on="id",
        right_on="order_id"
    )
    .merge(
        product_variants[["id", "product_id"]],
        left_on="product_variant_id",
        right_on="id"
    )
    .merge(
        products[["id", "name"]],
        left_on="product_id",
        right_on="id"
    )
)

#mantendo o necessário para a matriz de interação e chamando o nome do cliente apenas para melhor entendimento
df_final = df_combined_customers[["customer_id", "product_id", "name"]]
df_final_customer = df_final.merge(customers[["id", "legal_name"]], left_on="customer_id", right_on="id", suffixes=("_product", "_customer"))
df_final_customer = df_final_customer[["customer_id", "legal_name", "product_id", "name"]]

#mudando o nome das colunas para melhor entendimento
df_final_customer.columns = ["customer_id", "customer_name", "product_id", "product_name"]

#Eliminado compras repetitivas
df_final_customer = df_final_customer.drop_duplicates(subset=["customer_id", "product_id"])

#Criando matriz Cliente x produto pelos ids
matriz_cliente_produto = pd.crosstab(
    df_final_customer["customer_id"],
    df_final_customer["product_id"]
)
print(matriz_cliente_produto.shape)
#(número de clientes, número de produtos)

(2000, 500)


In [176]:
#2. Cálculo de Similaridade entre Produtos
#     a. Calcule a Similaridade de Cosseno (Cosine Similarity) entre os vetores dos produtos
#     b. A similaridade deve ser calculada produto × produto, com base nos clientes que compraram cada item

#Transpondo a matriz para termos produto x produto
matriz_produto_cliente = matriz_cliente_produto.T
print(matriz_produto_cliente.shape)

#Aplicando similaridade de Cosseno e transformando em novo dataframe para melhor visualização
similaridade = cosine_similarity(matriz_produto_cliente)
similaridade_df = pd.DataFrame(
    similaridade,
    index=matriz_produto_cliente.index,
    columns=matriz_produto_cliente.index
)
display(similaridade_df)


(500, 2000)


product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
product_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.226680,0.161690,0.145999,0.235263,0.194791,0.176263,0.113787,0.198287,0.127357,...,0.151973,0.150621,0.208979,0.120580,0.090764,0.160171,0.173645,0.229478,0.104979,0.150926
2,0.226680,1.000000,0.173133,0.189424,0.231390,0.203046,0.161226,0.128494,0.211475,0.094919,...,0.105911,0.159083,0.227560,0.122548,0.104824,0.189423,0.202057,0.183988,0.128031,0.147254
3,0.161690,0.173133,1.000000,0.120760,0.172844,0.173573,0.152272,0.109222,0.184563,0.090018,...,0.112532,0.135687,0.219806,0.100807,0.097202,0.157810,0.213454,0.183225,0.084095,0.145792
4,0.145999,0.189424,0.120760,1.000000,0.169926,0.151925,0.148931,0.080119,0.136513,0.091467,...,0.110063,0.145980,0.176573,0.108455,0.073203,0.169782,0.181804,0.164810,0.085152,0.156852
5,0.235263,0.231390,0.172844,0.169926,1.000000,0.188160,0.154636,0.134841,0.199603,0.105158,...,0.112039,0.131074,0.195149,0.116424,0.114970,0.183350,0.189613,0.221060,0.147357,0.183813
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,0.160171,0.189423,0.157810,0.169782,0.183350,0.169134,0.132640,0.125185,0.139615,0.090793,...,0.098024,0.147383,0.198052,0.134027,0.127451,1.000000,0.140538,0.187833,0.139702,0.179327
497,0.173645,0.202057,0.213454,0.181804,0.189613,0.194878,0.165901,0.142797,0.203595,0.135125,...,0.124833,0.163751,0.206910,0.159752,0.114374,0.140538,1.000000,0.188498,0.112101,0.192159
498,0.229478,0.183988,0.183225,0.164810,0.221060,0.205239,0.174199,0.118373,0.190998,0.114824,...,0.126478,0.141330,0.196696,0.125439,0.107297,0.187833,0.188498,1.000000,0.087368,0.166429


In [177]:
#3. Ranking de Produtos Similares
#     a. Considere o produto “Motor de Popa 1949” como item de referência
#     b. Gere um ranking com os nomes dos 5 produtos mais similares a ele
#     c. Desconsidere o próprio motor no ranking

#Identificando qual é o ID do Motor de Popa 1949
motor_de_popa_1949_id = df_final_customer[df_final_customer["product_name"] == "Motor de Popa 1949"]["product_id"].iloc[0]
#print(f"Similaridade do Motor de Popa 1949: \n {similaridade_df.loc[motor_de_popa_1949_id]}")
#id=180

#Pegando a similaridade do Motor de Popa 1949 com os demais produtos e ordenando do mais similar para o menos similar (5 primeiros)
ranking = (
    similaridade_df[motor_de_popa_1949_id]
    .drop(index=motor_de_popa_1949_id)      #Remove o próprio motor da similaridade
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
ranking = ranking.merge(
    products[["id", "name"]],
    left_on="product_id",
    right_on="id"
)
#mudando o nome das colunas para melhor entendimento
ranking.columns = ["product_id", "similarity", "id", "product_name"]

print(f"Ranking dos 5 produtos mais similares ao Motor de Popa 1949: \n {ranking}")

Ranking dos 5 produtos mais similares ao Motor de Popa 1949: 
    product_id  similarity   id        product_name
0         389    0.256553  389  Motor de Popa 5331
1         295    0.256239  295   Cabo Náutico 2105
2          75    0.255785   75    Vela Mestra 1913
3         337    0.239332  337   Cabo Náutico 9048
4          55    0.237744   55    GPS Plotter 6249


In [ ]:
#Qual é o nome do produto com MAIOR similaridade ao “Motor de Popa 1949”?
#Motor de Popa 5331

In [ ]:
#Como a matriz foi construída?
#A matriz foi criada com clientes nas linhas e produtos nas colunas. O valor é 1 quando o cliente comprou aquele produto pelo menos uma vez e 0 caso contrário. A quantidade comprada foi ignorada, considerando apenas presença ou ausência da compra.

#O que significa a similaridade de cosseno nesse contexto?
#Ela mede o quanto dois produtos têm comportamento de compra semelhante entre os clientes. Quanto mais próximo de 1, mais clientes em comum compraram os dois produtos; quanto mais próximo de 0, menor a relação entre eles.

#Uma limitação desse método de recomendação:
#Ele depende do histórico de compras. Produtos novos ou pouco comprados podem ter pouca informação para calcular a similaridade, dificultando recomendações precisas.